# 14_gold_multitarget_dataset_mac_safe.ipynb — Gold multitarget físico y de riesgo

Este notebook amplía el dataset Gold actual:

```text
gold/training_dataset/
```

y genera un nuevo dataset:

```text
gold/multitarget_training_dataset/
```

Objetivo:

```text
Preparar un Gold más completo para entrenar modelos que predigan varias variables marítimas y meteorológicas, no solo hs.
```

Variables objetivo futuras:

```text
Altura significativa de ola: hs
Altura máxima: hmax, si existe
Periodo pico / medio: tp, tm02
Dirección del oleaje: wave_direction
Swell: swell_height, swell_period, swell_direction
Viento: wind_speed, wind_direction, u10, v10
Corrientes: current_speed, current_direction
Nivel del mar: sea_level
```

Horizontes:

```text
+3h, +6h, +12h, +24h, +48h
```

Además, crea objetivos derivados:

```text
risk_general
risk_beach
risk_navigation
surf_score
surf_quality
```

Notas importantes:

```text
- Las direcciones se tratan como variables circulares usando sin/cos.
- Los riesgos y surf score son etiquetas derivadas con reglas físicas, no observaciones oficiales.
- Este Gold complementa al Gold v1, no lo sustituye.
```

> Versión adaptada para ejecución local en Mac/Windows/Linux, con modo RAM-safe.


## Celda 0 — Configurar entorno local / Colab

In [1]:
import sys
import platform
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Entorno detectado: Google Colab")
else:
    print("Entorno detectado: local")
    print("Sistema:", platform.platform())
    print("Python:", sys.version.split()[0])
    print("Home:", Path.home())

# En local no hace falta montar Google Drive.
# El proyecto debe estar sincronizado/copíado en tu Mac y se indicará la ruta en la celda 2.

Entorno detectado: local
Sistema: macOS-26.3.1-arm64-arm-64bit
Python: 3.11.15
Home: /Users/rauljimenez


## Celda 1 — Instalar librerías

In [2]:
%pip -q install pandas numpy pyarrow scikit-learn matplotlib tqdm psutil


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Celda 2 — Imports, rutas y configuración

In [3]:
from pathlib import Path
import os
import sys
import platform
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import shutil
import json
import gc
import warnings
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# CONFIGURACIÓN DE RUTA — MAC + GOOGLE DRIVE DESKTOP
# ---------------------------------------------------------------------
# Tus datasets están en Google Drive.
# En Mac, eso NO es /content/drive. Esa ruta solo existe dentro de Colab.
#
# En Mac, Google Drive suele estar en:
# /Users/tu_usuario/Library/CloudStorage/GoogleDrive-<cuenta>/My Drive/...
# o
# /Users/tu_usuario/Library/CloudStorage/GoogleDrive-<cuenta>/Mi unidad/...
#
# OPCIÓN A — recomendada:
# deja LOCAL_BASE_DIR = None y el notebook intentará encontrar gold/training_dataset.
#
# OPCIÓN B — manual:
# pega aquí la ruta exacta a la carpeta raíz que contiene gold/training_dataset:
#
# LOCAL_BASE_DIR = Path("/Users/rauljimenez/Library/CloudStorage/GoogleDrive-xxxxx/My Drive/AI Projects/DeepWave Canarias")
#
# OPCIÓN C — si tu carpeta de código es distinta de tus datos:
# puedes indicar directamente la ruta a gold/training_dataset:
#
# LOCAL_INPUT_GOLD_DIR = Path("/Users/rauljimenez/Library/CloudStorage/GoogleDrive-xxxxx/My Drive/AI Projects/DeepWave Canarias/gold/training_dataset")
#
LOCAL_BASE_DIR = Path.cwd().parent
LOCAL_INPUT_GOLD_DIR = None
RUNNING_IN_COLAB = "google.colab" in sys.modules


def has_training_dataset(path: Path) -> bool:
    path = Path(path).expanduser()
    return (path / "gold" / "training_dataset").exists()


def find_project_root_from_path(path: Path):
    path = Path(path).expanduser().resolve()
    candidates = [path] + list(path.parents)

    for p in candidates:
        if has_training_dataset(p):
            return p

    return None


def find_google_drive_roots():
    home = Path.home()
    cloud_root = home / "Library" / "CloudStorage"

    roots = []

    if cloud_root.exists():
        roots.extend([p for p in cloud_root.glob("GoogleDrive*") if p.is_dir()])

    # Fallback antiguo.
    old_google_drive = home / "Google Drive"
    if old_google_drive.exists():
        roots.append(old_google_drive)

    # Deduplicar.
    unique = []
    seen = set()

    for p in roots:
        rp = str(p.resolve())
        if rp not in seen:
            seen.add(rp)
            unique.append(p.resolve())

    return unique


def search_gold_training_dataset():
    """
    Busca gold/training_dataset en ubicaciones probables.
    Evita recorrer todo el disco.
    """
    home = Path.home()

    search_roots = [
        Path.cwd(),
        home / "Documents",
        home / "Desktop",
        home / "Development",
        home / "Projects",
        home / "AI Projects",
    ]

    search_roots.extend(find_google_drive_roots())

    found_training_dirs = []

    # Patrones típicos.
    patterns = [
        "AI Projects/DeepWave Canarias/gold/training_dataset",
        "AI Projects/deep-wave-canarias/gold/training_dataset",
        "DeepWave Canarias/gold/training_dataset",
        "deep-wave-canarias/gold/training_dataset",
        "*/AI Projects/DeepWave Canarias/gold/training_dataset",
        "*/AI Projects/deep-wave-canarias/gold/training_dataset",
        "*/*/AI Projects/DeepWave Canarias/gold/training_dataset",
        "*/*/AI Projects/deep-wave-canarias/gold/training_dataset",
        "*/*/*/gold/training_dataset",
        "*/*/*/*/gold/training_dataset",
        "*/*/*/*/*/gold/training_dataset",
    ]

    for root in search_roots:
        root = Path(root).expanduser()
        if not root.exists():
            continue

        for pattern in patterns:
            try:
                for p in root.glob(pattern):
                    if p.exists() and p.is_dir():
                        found_training_dirs.append(p.resolve())
            except Exception:
                pass

    # Deduplicar.
    unique = []
    seen = set()

    for p in found_training_dirs:
        rp = str(p)
        if rp not in seen:
            seen.add(rp)
            unique.append(p)

    return unique


def resolve_paths():
    if RUNNING_IN_COLAB:
        base_dir = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
        return base_dir, base_dir / "gold" / "training_dataset"

    # Si el usuario da directamente gold/training_dataset.
    if LOCAL_INPUT_GOLD_DIR is not None:
        input_dir = Path(LOCAL_INPUT_GOLD_DIR).expanduser()

        if not input_dir.exists():
            raise FileNotFoundError(f"LOCAL_INPUT_GOLD_DIR no existe: {input_dir}")

        if input_dir.name != "training_dataset" or input_dir.parent.name != "gold":
            raise FileNotFoundError(
                "LOCAL_INPUT_GOLD_DIR debe apuntar exactamente a gold/training_dataset.\n"
                f"Ruta indicada: {input_dir}"
            )

        base_dir = input_dir.parent.parent
        return base_dir.resolve(), input_dir.resolve()

    # Si el usuario da la raíz del proyecto.
    if LOCAL_BASE_DIR is not None:
        p = Path(LOCAL_BASE_DIR).expanduser()

        if has_training_dataset(p):
            return p.resolve(), (p / "gold" / "training_dataset").resolve()

        root = find_project_root_from_path(p) if p.exists() else None

        if root is not None:
            print("LOCAL_BASE_DIR apuntaba a una subcarpeta. Se usará como raíz:")
            print(root)
            return root.resolve(), (root / "gold" / "training_dataset").resolve()

        raise FileNotFoundError(
            "LOCAL_BASE_DIR no contiene gold/training_dataset y no se encontró en sus carpetas padre.\n"
            f"Ruta indicada: {p}\n\n"
            "Si tus datos están en Google Drive, pon LOCAL_BASE_DIR con la ruta local de Google Drive Desktop, "
            "o usa LOCAL_INPUT_GOLD_DIR apuntando directamente a gold/training_dataset."
        )

    # Variable de entorno opcional.
    env_base = os.environ.get("DEEPWAVE_BASE_DIR")
    if env_base:
        p = Path(env_base).expanduser()

        if has_training_dataset(p):
            return p.resolve(), (p / "gold" / "training_dataset").resolve()

    env_input = os.environ.get("DEEPWAVE_INPUT_GOLD_DIR")
    if env_input:
        input_dir = Path(env_input).expanduser()

        if input_dir.exists():
            base_dir = input_dir.parent.parent
            return base_dir.resolve(), input_dir.resolve()

    # Buscar automáticamente.
    found_training_dirs = search_gold_training_dataset()

    if len(found_training_dirs) == 1:
        input_dir = found_training_dirs[0]
        base_dir = input_dir.parent.parent
        print("Detectado gold/training_dataset automáticamente:")
        print(input_dir)
        return base_dir.resolve(), input_dir.resolve()

    if len(found_training_dirs) > 1:
        print("Se encontraron varias carpetas gold/training_dataset:")
        for i, p in enumerate(found_training_dirs):
            print(f"{i}: {p}")

        raise FileNotFoundError(
            "Hay varias carpetas posibles. Pon manualmente LOCAL_INPUT_GOLD_DIR en la celda 2 "
            "con una de las rutas mostradas arriba."
        )

    google_roots = find_google_drive_roots()

    print("No se encontró gold/training_dataset automáticamente.")
    print("\nRaíces de Google Drive detectadas en Mac:")
    if google_roots:
        for p in google_roots:
            print("-", p)
    else:
        print("- Ninguna. Abre Google Drive Desktop o sincroniza tu Drive.")

    raise FileNotFoundError(
        "No se encontró gold/training_dataset.\n\n"
        "Como tus datasets están en Drive, necesitas una de estas opciones:\n"
        "1) Instalar/abrir Google Drive Desktop y esperar a que sincronice.\n"
        "2) Poner LOCAL_BASE_DIR con la ruta local de Drive que contiene DeepWave Canarias.\n"
        "3) Poner LOCAL_INPUT_GOLD_DIR apuntando directamente a gold/training_dataset.\n\n"
        "Ejemplo:\n"
        'LOCAL_INPUT_GOLD_DIR = Path("/Users/rauljimenez/Library/CloudStorage/GoogleDrive-xxxxx/My Drive/AI Projects/DeepWave Canarias/gold/training_dataset")'
    )


BASE_DIR, INPUT_GOLD_DIR = resolve_paths()
GOLD_DIR = BASE_DIR / "gold"

OUTPUT_GOLD_DIR = GOLD_DIR / "multitarget_training_dataset"
REPORT_DIR = GOLD_DIR / "_quality_reports"
META_DIR = GOLD_DIR / "_metadata"

for d in [REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# CONFIGURACIÓN RAM-SAFE PARA MAC
# ---------------------------------------------------------------------
MAC_SAFE_MODE = True

HORIZONS_HOURS = [3, 6, 12, 24, 48]
REQUIRE_SPLIT_COLUMN = True
OVERWRITE_OUTPUT = True

if MAC_SAFE_MODE:
    LAG_HOURS = [1, 3, 6, 12, 24]
    ROLLING_WINDOWS = [3, 6, 12, 24]
    ROLLING_STATS = ["mean", "std"]
    MAX_DYNAMIC_FEATURES = 10
else:
    LAG_HOURS = [1, 3, 6, 12, 24, 48, 72]
    ROLLING_WINDOWS = [3, 6, 12, 24, 48, 72]
    ROLLING_STATS = ["mean", "std", "min", "max"]
    MAX_DYNAMIC_FEATURES = None

PHYSICAL_TARGET_SPECS = {
    "hs": ["simar_hs", "hs"],
    "hmax": ["simar_hmax", "hmax"],
    "tp": ["simar_tp", "tp"],
    "tm02": ["simar_tm02", "tm02"],
    "wave_direction": ["simar_wave_direction", "wave_direction"],
    "swell_height": ["simar_swell_height", "swell_height"],
    "swell_period": ["simar_swell_period", "swell_period"],
    "swell_direction": ["simar_swell_direction", "swell_direction"],
    "wind_wave_height": ["simar_wind_wave_height", "wind_wave_height"],
    "wind_wave_period": ["simar_wind_wave_period", "wind_wave_period"],
    "wind_wave_direction": ["simar_wind_wave_direction", "wind_wave_direction"],
    "wind_speed": ["simar_wind_speed", "wind_speed"],
    "wind_direction": ["simar_wind_direction", "wind_direction"],
    "u10": ["simar_u10", "u10"],
    "v10": ["simar_v10", "v10"],
    "current_speed": ["simar_current_speed", "current_speed"],
    "current_direction": ["simar_current_direction", "current_direction"],
    "sea_surface_temperature": ["simar_sst", "simar_sea_surface_temperature", "sea_surface_temperature", "sst"],
    "sea_surface_salinity": ["simar_salinity", "simar_sea_surface_salinity", "sea_surface_salinity"],
    "sea_level": ["redmar_sea_level", "sea_level"],
    "daily_tidal_range": ["redmar_daily_tidal_range", "daily_tidal_range"],
}

DIRECTION_TARGETS = [
    "wave_direction",
    "swell_direction",
    "wind_wave_direction",
    "wind_direction",
    "current_direction",
]

DYNAMIC_FEATURE_BASES = [
    "simar_hs",
    "simar_tp",
    "simar_tm02",
    "simar_wave_direction",
    "simar_swell_height",
    "simar_swell_period",
    "simar_swell_direction",
    "simar_wind_speed",
    "simar_wind_direction",
    "redmar_sea_level",
    "aemet_wind_speed",
    "aemet_temperature_air",
    "aemet_precipitation",
    "simar_current_speed",
]

if MAX_DYNAMIC_FEATURES is not None:
    DYNAMIC_FEATURE_BASES = DYNAMIC_FEATURE_BASES[:MAX_DYNAMIC_FEATURES]

RISK_THRESHOLDS = {
    "general": {
        "moderate_hs": 1.0,
        "high_hs": 2.0,
        "extreme_hs": 3.0,
    },
    "beach": {
        "moderate_hs": 1.0,
        "high_hs": 1.8,
        "extreme_hs": 2.7,
        "high_period": 12.0,
        "strong_wind": 10.0,
    },
    "navigation": {
        "moderate_hs": 1.2,
        "high_hs": 2.0,
        "extreme_hs": 3.0,
        "strong_wind": 10.0,
        "very_strong_wind": 14.0,
        "long_period": 11.0,
    },
}

SURF_SCORE_CONFIG = {
    "ideal_hs_min": 0.8,
    "ideal_hs_max": 2.5,
    "ideal_period_min": 9.0,
    "ideal_period_good": 12.0,
    "bad_wind_speed": 10.0,
    "very_bad_wind_speed": 14.0,
}

print("BASE_DIR:", BASE_DIR)
print("INPUT_GOLD_DIR:", INPUT_GOLD_DIR, "existe:", INPUT_GOLD_DIR.exists())
print("OUTPUT_GOLD_DIR:", OUTPUT_GOLD_DIR)
print("MAC_SAFE_MODE:", MAC_SAFE_MODE)
print("LAG_HOURS:", LAG_HOURS)
print("ROLLING_WINDOWS:", ROLLING_WINDOWS)
print("ROLLING_STATS:", ROLLING_STATS)
print("DYNAMIC_FEATURE_BASES:", DYNAMIC_FEATURE_BASES)

if not INPUT_GOLD_DIR.exists():
    raise FileNotFoundError(
        f"No existe INPUT_GOLD_DIR: {INPUT_GOLD_DIR}"
    )

BASE_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias
INPUT_GOLD_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/training_dataset existe: True
OUTPUT_GOLD_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/multitarget_training_dataset
MAC_SAFE_MODE: True
LAG_HOURS: [1, 3, 6, 12, 24]
ROLLING_WINDOWS: [3, 6, 12, 24]
ROLLING_STATS: ['mean', 'std']
DYNAMIC_FEATURE_BASES: ['simar_hs', 'simar_tp', 'simar_tm02', 'simar_wave_direction', 'simar_swell_height', 'simar_swell_period', 'simar_swell_direction', 'simar_wind_speed', 'simar_wind_direction', 'redmar_sea_level']


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Celda 3 — Funciones auxiliares

In [4]:
def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def memory_report(name, df):
    mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{name}: shape={df.shape}, memoria≈{mb:.1f} MB")


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def find_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def circular_sin_deg(deg):
    x = pd.to_numeric(deg, errors="coerce")
    return np.sin(np.deg2rad(x))


def circular_cos_deg(deg):
    x = pd.to_numeric(deg, errors="coerce")
    return np.cos(np.deg2rad(x))


def angular_diff_deg(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.abs((a - b + 180) % 360 - 180)


def direction_to_uv(speed, direction_deg):
    speed = pd.to_numeric(speed, errors="coerce")
    direction_deg = pd.to_numeric(direction_deg, errors="coerce")
    u = speed * np.sin(np.deg2rad(direction_deg))
    v = speed * np.cos(np.deg2rad(direction_deg))
    return u, v


def coast_orientation_to_degrees(value):
    if pd.isna(value):
        return np.nan
    v = str(value).strip().upper()
    mapping = {"N": 0, "NE": 45, "E": 90, "SE": 135, "S": 180, "SW": 225, "W": 270, "NW": 315}
    return mapping.get(v, np.nan)


def quality_flag_from_target(series):
    return np.where(pd.Series(series).notna(), 0, 1).astype("int8")


def optimize_numeric_dtypes(df):
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
        elif pd.api.types.is_integer_dtype(df[c]):
            if c.endswith("_flag") or c.startswith("target_risk") or c.startswith("is_trainable_"):
                df[c] = df[c].astype("Int16")
    return df


def assign_risk_general(hs):
    hs = pd.to_numeric(hs, errors="coerce")
    risk = pd.Series(np.nan, index=hs.index, dtype="float64")
    risk.loc[hs < RISK_THRESHOLDS["general"]["moderate_hs"]] = 0
    risk.loc[(hs >= RISK_THRESHOLDS["general"]["moderate_hs"]) & (hs < RISK_THRESHOLDS["general"]["high_hs"])] = 1
    risk.loc[(hs >= RISK_THRESHOLDS["general"]["high_hs"]) & (hs < RISK_THRESHOLDS["general"]["extreme_hs"])] = 2
    risk.loc[hs >= RISK_THRESHOLDS["general"]["extreme_hs"]] = 3
    return risk.astype("Int16")


def assign_risk_beach(hs, period=None, wind_speed=None, exposure=None):
    hs = pd.to_numeric(hs, errors="coerce")
    period = pd.to_numeric(period, errors="coerce") if period is not None else pd.Series(np.nan, index=hs.index)
    wind_speed = pd.to_numeric(wind_speed, errors="coerce") if wind_speed is not None else pd.Series(np.nan, index=hs.index)
    exposure = pd.to_numeric(exposure, errors="coerce") if exposure is not None else pd.Series(0, index=hs.index)

    risk = pd.Series(np.nan, index=hs.index, dtype="float64")
    risk.loc[hs < RISK_THRESHOLDS["beach"]["moderate_hs"]] = 0
    risk.loc[(hs >= RISK_THRESHOLDS["beach"]["moderate_hs"]) & (hs < RISK_THRESHOLDS["beach"]["high_hs"])] = 1
    risk.loc[(hs >= RISK_THRESHOLDS["beach"]["high_hs"]) & (hs < RISK_THRESHOLDS["beach"]["extreme_hs"])] = 2
    risk.loc[hs >= RISK_THRESHOLDS["beach"]["extreme_hs"]] = 3

    boost_period = (period >= RISK_THRESHOLDS["beach"]["high_period"]) & (hs >= 1.2)
    boost_wind = wind_speed >= RISK_THRESHOLDS["beach"]["strong_wind"]
    boost_exposure = (exposure >= 0.5) & (hs >= 1.0)
    boost = boost_period.astype("float64").fillna(0) + boost_wind.astype("float64").fillna(0) + boost_exposure.astype("float64").fillna(0)
    risk = risk + np.where(boost >= 2, 1, 0)
    return risk.clip(0, 3).astype("Int16")


def assign_risk_navigation(hs, period=None, wind_speed=None):
    hs = pd.to_numeric(hs, errors="coerce")
    period = pd.to_numeric(period, errors="coerce") if period is not None else pd.Series(np.nan, index=hs.index)
    wind_speed = pd.to_numeric(wind_speed, errors="coerce") if wind_speed is not None else pd.Series(np.nan, index=hs.index)

    risk = pd.Series(np.nan, index=hs.index, dtype="float64")
    risk.loc[hs < RISK_THRESHOLDS["navigation"]["moderate_hs"]] = 0
    risk.loc[(hs >= RISK_THRESHOLDS["navigation"]["moderate_hs"]) & (hs < RISK_THRESHOLDS["navigation"]["high_hs"])] = 1
    risk.loc[(hs >= RISK_THRESHOLDS["navigation"]["high_hs"]) & (hs < RISK_THRESHOLDS["navigation"]["extreme_hs"])] = 2
    risk.loc[hs >= RISK_THRESHOLDS["navigation"]["extreme_hs"]] = 3

    risk = risk + (wind_speed >= RISK_THRESHOLDS["navigation"]["strong_wind"]).astype("float64").fillna(0)
    risk = risk + (wind_speed >= RISK_THRESHOLDS["navigation"]["very_strong_wind"]).astype("float64").fillna(0)
    risk = risk + ((period >= RISK_THRESHOLDS["navigation"]["long_period"]) & (hs >= 1.5)).astype("float64").fillna(0)
    return risk.clip(0, 3).astype("Int16")


def surf_score_from_conditions(hs, period=None, wind_speed=None, offshore_score=None):
    hs = pd.to_numeric(hs, errors="coerce")
    period = pd.to_numeric(period, errors="coerce") if period is not None else pd.Series(np.nan, index=hs.index)
    wind_speed = pd.to_numeric(wind_speed, errors="coerce") if wind_speed is not None else pd.Series(np.nan, index=hs.index)
    offshore_score = pd.to_numeric(offshore_score, errors="coerce") if offshore_score is not None else pd.Series(0.5, index=hs.index)

    score = pd.Series(0.0, index=hs.index, dtype="float64")
    score += np.where(hs.between(SURF_SCORE_CONFIG["ideal_hs_min"], SURF_SCORE_CONFIG["ideal_hs_max"]), 3.0, 0.0)
    score += np.where(hs.between(0.5, 3.0), 1.0, 0.0)
    score -= np.where(hs < 0.4, 2.0, 0.0)
    score -= np.where(hs > 3.5, 1.5, 0.0)
    score += np.where(period >= SURF_SCORE_CONFIG["ideal_period_min"], 2.0, 0.0)
    score += np.where(period >= SURF_SCORE_CONFIG["ideal_period_good"], 1.0, 0.0)
    score += np.where(wind_speed <= 5.0, 1.0, 0.0)
    score -= np.where(wind_speed >= SURF_SCORE_CONFIG["bad_wind_speed"], 1.0, 0.0)
    score -= np.where(wind_speed >= SURF_SCORE_CONFIG["very_bad_wind_speed"], 1.0, 0.0)
    score += offshore_score.fillna(0.5).clip(0, 1) * 2.0
    return score.clip(0, 10).astype("float32")


def surf_quality_label(score):
    score = pd.to_numeric(score, errors="coerce")
    out = pd.Series(pd.NA, index=score.index, dtype="string")
    out.loc[score < 2] = "poor"
    out.loc[(score >= 2) & (score < 4)] = "fair"
    out.loc[(score >= 4) & (score < 6)] = "good"
    out.loc[(score >= 6) & (score < 8)] = "very_good"
    out.loc[score >= 8] = "epic"
    return out

## Celda 4 — Cargar Gold v1

In [5]:
dataset = ds.dataset(str(INPUT_GOLD_DIR), format="parquet", partitioning="hive")
gold = dataset.to_table().to_pandas()

gold["timestamp"] = ensure_utc(gold["timestamp"])

if REQUIRE_SPLIT_COLUMN and "split" not in gold.columns:
    raise ValueError("El Gold v1 no tiene columna split.")

if "zona_id" not in gold.columns:
    raise ValueError("El Gold v1 no tiene zona_id.")

gold["zona_id"] = gold["zona_id"].astype(str)

gold["isla"] = gold["isla"].astype("string").fillna("UNKNOWN") if "isla" in gold.columns else "UNKNOWN"

gold["year"] = gold["timestamp"].dt.year.astype("int16")
gold["month"] = gold["timestamp"].dt.month.astype("int8")
gold["dayofweek"] = gold["timestamp"].dt.dayofweek.astype("int8")
gold["hour"] = gold["timestamp"].dt.hour.astype("int8")

gold = gold.sort_values(["zona_id", "timestamp"]).reset_index(drop=True)

print("Gold v1 cargado:")
memory_report("gold", gold)
print("Periodo:", gold["timestamp"].min(), "→", gold["timestamp"].max())
print("Zonas:", gold["zona_id"].nunique())
print("Columnas:", len(gold.columns))
display(gold.head())

Gold v1 cargado:
gold: shape=(983328, 142), memoria≈1130.0 MB
Periodo: 2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
Zonas: 14
Columnas: 142


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,target_risk_24h,gold_dataset_version,target_source,risk_threshold_low_max,risk_threshold_moderate_max,risk_threshold_high_max,has_any_target,is_trainable_target_3h,split,year
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2025
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2025
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2025
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2025
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2025


## Celda 5 — Detectar variables disponibles para targets físicos

In [6]:
target_source_map = {}

for target_name, candidates in PHYSICAL_TARGET_SPECS.items():
    source_col = find_first_existing(gold, candidates)
    if source_col is not None:
        # No incluir columnas casi totalmente nulas.
        missing_pct = float(gold[source_col].isna().mean() * 100)
        if missing_pct < 99.5:
            target_source_map[target_name] = source_col

target_source_df = pd.DataFrame(
    [
        {
            "target_name": k,
            "source_col": v,
            "missing_pct_source": float(gold[v].isna().mean() * 100),
            "nunique_source": int(gold[v].nunique(dropna=True)),
        }
        for k, v in target_source_map.items()
    ]
).sort_values("target_name")

print("Targets físicos disponibles:", len(target_source_map))
display(target_source_df)

target_source_df.to_csv(REPORT_DIR / "quality_gold_multitarget_source_variables.csv", index=False)
save_json(target_source_map, META_DIR / "gold_multitarget_target_source_map.json")

if "hs" not in target_source_map:
    raise ValueError("No se encontró variable fuente para hs. Es imprescindible.")

Targets físicos disponibles: 18


,target_name,source_col,missing_pct_source,nunique_source
13,current_direction,simar_current_direction,9.609205,36000
12,current_speed,simar_current_speed,9.609205,1084
17,daily_tidal_range,redmar_daily_tidal_range,30.402470,5619
0,hs,simar_hs,0.000000,735
16,sea_level,redmar_sea_level,30.596200,3015
15,sea_surface_salinity,simar_salinity,9.030761,1433
14,sea_surface_temperature,simar_sst,9.030761,9512
6,swell_direction,simar_swell_direction,0.000000,361
4,swell_height,simar_swell_height,0.000000,604
5,swell_period,simar_swell_period,0.000000,2002


## Celda 6 — Crear features circulares, vectoriales e interacciones físicas actuales

In [7]:
if "orientacion_costa" in gold.columns:
    gold["coast_orientation_deg"] = gold["orientacion_costa"].map(coast_orientation_to_degrees).astype("float32")
else:
    gold["coast_orientation_deg"] = np.nan

exposure_cols = [
    c for c in ["exposicion_norte", "exposicion_oeste", "exposicion_este", "exposicion_swell_nw", "exposicion_swell_ne"]
    if c in gold.columns
]

gold["coast_exposure_score"] = (
    gold[exposure_cols].apply(pd.to_numeric, errors="coerce").fillna(0).mean(axis=1).astype("float32")
    if exposure_cols else 0.5
)

direction_current_sources = {
    "wave_direction": find_first_existing(gold, ["simar_wave_direction", "wave_direction"]),
    "swell_direction": find_first_existing(gold, ["simar_swell_direction", "swell_direction"]),
    "wind_direction": find_first_existing(gold, ["simar_wind_direction", "wind_direction"]),
    "current_direction": find_first_existing(gold, ["simar_current_direction", "current_direction"]),
}

for name, col in direction_current_sources.items():
    if col is None:
        continue
    gold[f"{col}_sin"] = circular_sin_deg(gold[col]).astype("float32")
    gold[f"{col}_cos"] = circular_cos_deg(gold[col]).astype("float32")
    gold[f"{col}_coast_angle_diff"] = angular_diff_deg(gold[col], gold["coast_orientation_deg"]).astype("float32")
    gold[f"{col}_coast_alignment"] = np.cos(np.deg2rad(gold[f"{col}_coast_angle_diff"])).astype("float32")

wind_speed_col = find_first_existing(gold, ["simar_wind_speed", "wind_speed"])
wind_dir_col = find_first_existing(gold, ["simar_wind_direction", "wind_direction"])

if wind_speed_col is not None and wind_dir_col is not None:
    gold["wind_u_from_speed_dir"], gold["wind_v_from_speed_dir"] = direction_to_uv(gold[wind_speed_col], gold[wind_dir_col])
    gold["wind_u_from_speed_dir"] = gold["wind_u_from_speed_dir"].astype("float32")
    gold["wind_v_from_speed_dir"] = gold["wind_v_from_speed_dir"].astype("float32")

current_speed_col = find_first_existing(gold, ["simar_current_speed", "current_speed"])
current_dir_col = find_first_existing(gold, ["simar_current_direction", "current_direction"])

if current_speed_col is not None and current_dir_col is not None:
    gold["current_u_from_speed_dir"], gold["current_v_from_speed_dir"] = direction_to_uv(gold[current_speed_col], gold[current_dir_col])
    gold["current_u_from_speed_dir"] = gold["current_u_from_speed_dir"].astype("float32")
    gold["current_v_from_speed_dir"] = gold["current_v_from_speed_dir"].astype("float32")

hs_col = target_source_map.get("hs")
tp_col = target_source_map.get("tp") or target_source_map.get("tm02")

if hs_col is not None:
    hs = pd.to_numeric(gold[hs_col], errors="coerce")
    gold["wave_energy_proxy"] = (hs ** 2).astype("float32")
    if tp_col is not None:
        period = pd.to_numeric(gold[tp_col], errors="coerce")
        gold["wave_power_proxy"] = (hs ** 2 * period).astype("float32")
        gold["wave_steepness_proxy"] = (hs / period.replace(0, np.nan)).astype("float32")

wave_dir_col = direction_current_sources.get("wave_direction")
if wind_dir_col is not None and wave_dir_col is not None:
    gold["wind_wave_angle_diff"] = angular_diff_deg(gold[wind_dir_col], gold[wave_dir_col]).astype("float32")

print("Features físicas actuales añadidas.")
print("Columnas totales:", len(gold.columns))
display(gold.head())

Features físicas actuales añadidas.
Columnas totales: 168


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,simar_current_direction_coast_angle_diff,simar_current_direction_coast_alignment,wind_u_from_speed_dir,wind_v_from_speed_dir,current_u_from_speed_dir,current_v_from_speed_dir,wave_energy_proxy,wave_power_proxy,wave_steepness_proxy,wind_wave_angle_diff
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,120.610001,-0.509192,NaN,NaN,0.157849,0.266802,0.9409,8.562190,0.106593,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,118.720001,-0.480530,NaN,NaN,0.130704,0.238538,0.8836,8.844836,0.093906,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,113.250000,-0.394744,NaN,NaN,0.097896,0.227860,0.8281,8.289281,0.090909,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,105.779999,-0.271944,NaN,NaN,0.064995,0.229993,0.8100,9.809100,0.074319,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,98.529999,-0.148327,NaN,NaN,0.035895,0.239323,0.8100,9.809100,0.074319,NaN


## Celda 7 — Crear lags, tendencias y rolling features adicionales

In [8]:
existing_cols_before = set(gold.columns)

dynamic_cols = [c for c in DYNAMIC_FEATURE_BASES if c in gold.columns]

print("Variables dinámicas para lags/rolling:", dynamic_cols)
print("Modo RAM-safe:", MAC_SAFE_MODE)

gold = gold.sort_values(["zona_id", "timestamp"]).reset_index(drop=True)

new_feature_frames = []

for col in tqdm(dynamic_cols, desc="Creando lags y rolling RAM-safe"):
    feature_dict = {}
    base = pd.to_numeric(gold[col], errors="coerce").astype("float32")

    grouped = gold.groupby("zona_id", sort=False)[col]

    # Lags.
    for lag in LAG_HOURS:
        new_col = f"{col}_lag_{lag}h_mt"
        feature_dict[new_col] = grouped.shift(lag).astype("float32")

    # Diferencias respecto a lags.
    for lag in [1, 3, 6, 12, 24]:
        if lag in LAG_HOURS:
            lag_col = f"{col}_lag_{lag}h_mt"
            diff_col = f"{col}_diff_{lag}h_mt"
            feature_dict[diff_col] = (base - feature_dict[lag_col]).astype("float32")

    # Rollings. Se calculan por variable y se concatena al final para evitar
    # fragmentación de memoria por cientos de asignaciones directas.
    for window in ROLLING_WINDOWS:
        min_periods = max(2, window // 3)

        if "mean" in ROLLING_STATS:
            feature_dict[f"{col}_roll_mean_{window}h_mt"] = (
                gold.groupby("zona_id", sort=False)[col]
                .transform(lambda s: pd.to_numeric(s, errors="coerce").rolling(window, min_periods=min_periods).mean())
                .astype("float32")
            )

        if "std" in ROLLING_STATS:
            feature_dict[f"{col}_roll_std_{window}h_mt"] = (
                gold.groupby("zona_id", sort=False)[col]
                .transform(lambda s: pd.to_numeric(s, errors="coerce").rolling(window, min_periods=min_periods).std())
                .astype("float32")
            )

        if "min" in ROLLING_STATS:
            feature_dict[f"{col}_roll_min_{window}h_mt"] = (
                gold.groupby("zona_id", sort=False)[col]
                .transform(lambda s: pd.to_numeric(s, errors="coerce").rolling(window, min_periods=min_periods).min())
                .astype("float32")
            )

        if "max" in ROLLING_STATS:
            feature_dict[f"{col}_roll_max_{window}h_mt"] = (
                gold.groupby("zona_id", sort=False)[col]
                .transform(lambda s: pd.to_numeric(s, errors="coerce").rolling(window, min_periods=min_periods).max())
                .astype("float32")
            )

    new_feature_frames.append(pd.DataFrame(feature_dict, index=gold.index))

    # Limpieza intermedia.
    del feature_dict
    gc.collect()

if new_feature_frames:
    new_features = pd.concat(new_feature_frames, axis=1)
    gold = pd.concat([gold, new_features], axis=1)
    del new_features
    del new_feature_frames
    gc.collect()

new_cols = sorted(set(gold.columns) - existing_cols_before)
print("Nuevas columnas creadas:", len(new_cols))
display(pd.DataFrame({"new_columns": new_cols}).head(100))

memory_report("gold con lags/rolling multitarget", gold)

Variables dinámicas para lags/rolling: ['simar_hs', 'simar_tp', 'simar_tm02', 'simar_wave_direction', 'simar_swell_height', 'simar_swell_period', 'simar_swell_direction', 'simar_wind_speed', 'simar_wind_direction', 'redmar_sea_level']
Modo RAM-safe: True


Creando lags y rolling RAM-safe: 100%|██████████| 10/10 [00:03<00:00,  3.18it/s]

Nuevas columnas creadas: 180


,new_columns
0,redmar_sea_level_diff_12h_mt
1,redmar_sea_level_diff_1h_mt
2,redmar_sea_level_diff_24h_mt
3,redmar_sea_level_diff_3h_mt
4,redmar_sea_level_diff_6h_mt
...,...
95,simar_tm02_lag_12h_mt
96,simar_tm02_lag_1h_mt
97,simar_tm02_lag_24h_mt
98,simar_tm02_lag_3h_mt


gold con lags/rolling multitarget: shape=(983328, 348), memoria≈1902.8 MB


## Celda 8 — Construir targets físicos futuros por shift temporal RAM-safe

In [9]:
gold = gold.sort_values(["zona_id", "timestamp"]).reset_index(drop=True)

target_created_cols = []

# Crear targets futuros por zona con shift(-h).
# Para evitar errores cuando haya huecos temporales, se valida que la fila futura
# corresponda exactamente a timestamp + h horas. Si no, se marca NaN.
grouped_ts = gold.groupby("zona_id", sort=False)["timestamp"]

for horizon in tqdm(HORIZONS_HOURS, desc="Creando targets futuros RAM-safe"):
    expected_future_ts = gold["timestamp"] + pd.Timedelta(hours=horizon)
    actual_future_ts = grouped_ts.shift(-horizon)
    valid_future = actual_future_ts.eq(expected_future_ts)

    gold[f"target_time_valid_{horizon}h"] = valid_future.fillna(False).astype("bool")

    for target_name, source_col in target_source_map.items():
        target_col = f"target_{target_name}_{horizon}h"

        shifted = gold.groupby("zona_id", sort=False)[source_col].shift(-horizon)
        shifted = pd.to_numeric(shifted, errors="coerce").astype("float32")
        shifted.loc[~valid_future] = np.nan

        gold[target_col] = shifted
        target_created_cols.append(target_col)

        gold[f"{target_col}_flag"] = quality_flag_from_target(gold[target_col])

        # Seno/coseno para direcciones futuras.
        if target_name in DIRECTION_TARGETS:
            gold[f"{target_col}_sin"] = circular_sin_deg(gold[target_col]).astype("float32")
            gold[f"{target_col}_cos"] = circular_cos_deg(gold[target_col]).astype("float32")

            gold[f"{target_col}_sin_flag"] = quality_flag_from_target(gold[f"{target_col}_sin"])
            gold[f"{target_col}_cos_flag"] = quality_flag_from_target(gold[f"{target_col}_cos"])

    del expected_future_ts, actual_future_ts, valid_future
    gc.collect()

print("Targets físicos futuros creados:", len(target_created_cols))
display(pd.DataFrame({"target_col": target_created_cols}).head(100))

memory_report("gold con targets físicos", gold)

Creando targets futuros RAM-safe: 100%|██████████| 5/5 [00:01<00:00,  2.53it/s]

Targets físicos futuros creados: 90


,target_col
0,target_hs_3h
1,target_tp_3h
2,target_tm02_3h
3,target_wave_direction_3h
4,target_swell_height_3h
...,...
85,target_current_direction_48h
86,target_sea_surface_temperature_48h
87,target_sea_surface_salinity_48h
88,target_sea_level_48h


gold con targets físicos: shape=(983328, 609), memoria≈2487.0 MB


## Celda 9 — Crear targets derivados: riesgo playa, navegación y surf score

In [10]:
derived_target_cols = []

for horizon in tqdm(HORIZONS_HOURS, desc="Creando targets derivados"):
    hs_target = f"target_hs_{horizon}h"
    tp_target = f"target_tp_{horizon}h" if f"target_tp_{horizon}h" in gold.columns else None
    if tp_target is None and f"target_tm02_{horizon}h" in gold.columns:
        tp_target = f"target_tm02_{horizon}h"

    wind_speed_target = f"target_wind_speed_{horizon}h" if f"target_wind_speed_{horizon}h" in gold.columns else None

    if hs_target not in gold.columns:
        print(f"AVISO: no existe {hs_target}; se omiten riesgos h={horizon}.")
        continue

    hs = gold[hs_target]
    period = gold[tp_target] if tp_target is not None else None
    wind_speed = gold[wind_speed_target] if wind_speed_target is not None else None
    exposure = gold["coast_exposure_score"] if "coast_exposure_score" in gold.columns else None

    gold[f"target_risk_general_{horizon}h"] = assign_risk_general(hs)
    gold[f"target_risk_general_{horizon}h_flag"] = quality_flag_from_target(gold[f"target_risk_general_{horizon}h"])

    gold[f"target_risk_beach_{horizon}h"] = assign_risk_beach(hs=hs, period=period, wind_speed=wind_speed, exposure=exposure)
    gold[f"target_risk_beach_{horizon}h_flag"] = quality_flag_from_target(gold[f"target_risk_beach_{horizon}h"])

    gold[f"target_risk_navigation_{horizon}h"] = assign_risk_navigation(hs=hs, period=period, wind_speed=wind_speed)
    gold[f"target_risk_navigation_{horizon}h_flag"] = quality_flag_from_target(gold[f"target_risk_navigation_{horizon}h"])

    wave_dir_target = f"target_wave_direction_{horizon}h" if f"target_wave_direction_{horizon}h" in gold.columns else None
    wind_dir_target = f"target_wind_direction_{horizon}h" if f"target_wind_direction_{horizon}h" in gold.columns else None

    offshore_score = None
    if wave_dir_target is not None and "coast_orientation_deg" in gold.columns:
        wave_coast_diff = angular_diff_deg(gold[wave_dir_target], gold["coast_orientation_deg"])
        wave_alignment = pd.Series(np.cos(np.deg2rad(wave_coast_diff)), index=gold.index).clip(-1, 1)
        offshore_score = (wave_alignment + 1) / 2

    if wind_dir_target is not None and "coast_orientation_deg" in gold.columns:
        wind_coast_diff = angular_diff_deg(gold[wind_dir_target], gold["coast_orientation_deg"])
        wind_offshore = pd.Series(wind_coast_diff / 180.0, index=gold.index).clip(0, 1)
        offshore_score = wind_offshore if offshore_score is None else (offshore_score + wind_offshore) / 2

    gold[f"target_surf_score_{horizon}h"] = surf_score_from_conditions(
        hs=hs,
        period=period,
        wind_speed=wind_speed,
        offshore_score=offshore_score,
    )
    gold[f"target_surf_quality_{horizon}h"] = surf_quality_label(gold[f"target_surf_score_{horizon}h"])
    gold[f"target_surf_score_{horizon}h_flag"] = quality_flag_from_target(gold[f"target_surf_score_{horizon}h"])

    derived_target_cols.extend([
        f"target_risk_general_{horizon}h",
        f"target_risk_beach_{horizon}h",
        f"target_risk_navigation_{horizon}h",
        f"target_surf_score_{horizon}h",
        f"target_surf_quality_{horizon}h",
    ])

print("Targets derivados creados:", len(derived_target_cols))
display(pd.DataFrame({"derived_target": derived_target_cols}))

Creando targets derivados: 100%|██████████| 5/5 [00:00<00:00,  7.49it/s]

Targets derivados creados: 25


,derived_target
0,target_risk_general_3h
1,target_risk_beach_3h
2,target_risk_navigation_3h
3,target_surf_score_3h
4,target_surf_quality_3h
5,target_risk_general_6h
6,target_risk_beach_6h
7,target_risk_navigation_6h
8,target_surf_score_6h
9,target_surf_quality_6h


## Celda 10 — Crear flags de trainability y resumen de targets

In [11]:
target_cols = [c for c in gold.columns if c.startswith("target_") and not c.endswith("_flag")]

for horizon in HORIZONS_HOURS:
    required = [f"target_hs_{horizon}h"]
    optional_physical = [
        f"target_tp_{horizon}h",
        f"target_wave_direction_{horizon}h",
        f"target_wind_speed_{horizon}h",
        f"target_wind_direction_{horizon}h",
    ]
    available_optional = [c for c in optional_physical if c in gold.columns]

    if all(c in gold.columns for c in required):
        gold[f"is_trainable_physical_{horizon}h"] = gold[required].notna().all(axis=1)

        if available_optional:
            gold[f"is_trainable_multitarget_{horizon}h"] = gold[required + available_optional].notna().all(axis=1)
        else:
            gold[f"is_trainable_multitarget_{horizon}h"] = gold[f"is_trainable_physical_{horizon}h"]
    else:
        gold[f"is_trainable_physical_{horizon}h"] = False
        gold[f"is_trainable_multitarget_{horizon}h"] = False

    risk_cols_h = [
        f"target_risk_general_{horizon}h",
        f"target_risk_beach_{horizon}h",
        f"target_risk_navigation_{horizon}h",
    ]
    risk_cols_h = [c for c in risk_cols_h if c in gold.columns]

    if risk_cols_h:
        gold[f"is_trainable_risk_{horizon}h"] = gold[risk_cols_h].notna().all(axis=1)
    else:
        gold[f"is_trainable_risk_{horizon}h"] = False


# ---------------------------------------------------------------------
# Resumen robusto de targets.
# IMPORTANTE:
# pandas considera boolean como numérico, pero numpy no permite quantile
# sobre booleanos. Por eso separamos bool, numérico real y categórico.
# ---------------------------------------------------------------------
target_summary_rows = []

for c in target_cols:
    s = gold[c]

    row = {
        "target_col": c,
        "dtype": str(s.dtype),
        "missing_pct": float(s.isna().mean() * 100),
        "non_null_rows": int(s.notna().sum()),
        "nunique": int(s.nunique(dropna=True)),
    }

    is_bool = pd.api.types.is_bool_dtype(s)
    is_numeric = pd.api.types.is_numeric_dtype(s) and not is_bool

    if is_bool:
        # Para flags/booleans, resumen simple sin cuantiles.
        numeric = s.astype("float32")
        row.update({
            "mean": float(numeric.mean()),
            "std": float(numeric.std()),
            "min": float(numeric.min()),
            "p05": np.nan,
            "p50": float(numeric.median()),
            "p95": np.nan,
            "max": float(numeric.max()),
        })

    elif is_numeric:
        numeric = pd.to_numeric(s, errors="coerce").astype("float64")
        numeric_non_null = numeric.dropna()

        if len(numeric_non_null):
            row.update({
                "mean": float(numeric_non_null.mean()),
                "std": float(numeric_non_null.std()),
                "min": float(numeric_non_null.min()),
                "p05": float(numeric_non_null.quantile(0.05)),
                "p50": float(numeric_non_null.quantile(0.50)),
                "p95": float(numeric_non_null.quantile(0.95)),
                "max": float(numeric_non_null.max()),
            })
        else:
            row.update({
                "mean": np.nan,
                "std": np.nan,
                "min": np.nan,
                "p05": np.nan,
                "p50": np.nan,
                "p95": np.nan,
                "max": np.nan,
            })

    else:
        # Targets categóricos, por ejemplo surf_quality.
        top_values = s.astype("string").value_counts(dropna=True).head(5).to_dict()
        row.update({
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "p05": np.nan,
            "p50": np.nan,
            "p95": np.nan,
            "max": np.nan,
            "top_values": json.dumps(top_values, ensure_ascii=False),
        })

    target_summary_rows.append(row)

target_summary = pd.DataFrame(target_summary_rows).sort_values("target_col")
target_summary.to_csv(REPORT_DIR / "quality_gold_multitarget_target_summary.csv", index=False)
display(target_summary.head(100))


# ---------------------------------------------------------------------
# Trainability summary robusto.
# Convertimos flags booleanos a float para calcular porcentajes.
# ---------------------------------------------------------------------
trainability_cols = [c for c in gold.columns if c.startswith("is_trainable_")]

trainability_work = gold[["split"] + trainability_cols].copy()

for c in trainability_cols:
    trainability_work[c] = trainability_work[c].astype("float32")

trainability_summary = (
    trainability_work
    .groupby("split", as_index=False)
    .mean(numeric_only=True)
)

for c in trainability_cols:
    trainability_summary[c] = trainability_summary[c] * 100

trainability_summary.to_csv(REPORT_DIR / "quality_gold_multitarget_trainability_summary.csv", index=False)
display(trainability_summary)

,target_col,dtype,missing_pct,non_null_rows,nunique,mean,std,min,p05,p50,p95,max,top_values
80,target_current_direction_12h,float32,9.916630,885815,36000,197.450203,98.378751,0.000000,21.240000,213.899994,341.152997,359.98999,NaN
82,target_current_direction_12h_cos,float32,9.916630,885815,31861,-0.060692,0.696015,-1.000000,-0.989601,-0.134332,0.984838,1.00000,NaN
81,target_current_direction_12h_sin,float32,9.916630,885815,28927,-0.176895,0.693245,-1.000000,-0.993923,-0.332820,0.977783,1.00000,NaN
106,target_current_direction_24h,float32,10.222937,882803,36000,197.419591,98.377061,0.000000,21.240000,213.850006,341.140015,359.98999,NaN
108,target_current_direction_24h_cos,float32,10.222937,882803,31861,-0.060806,0.696019,-1.000000,-0.989626,-0.134505,0.984838,1.00000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,target_swell_period_12h,float32,0.327053,980112,2002,-52.550181,780.761177,-9999.900391,4.400000,8.530000,13.660000,23.93000,NaN
94,target_swell_period_24h,float32,0.654105,976896,2002,-52.722163,781.842964,-9999.900391,4.400000,8.530000,13.660000,23.93000,NaN
16,target_swell_period_3h,float32,0.081763,982524,2002,-52.490999,780.389119,-9999.900391,4.400000,8.530000,13.660000,23.93000,NaN
121,target_swell_period_48h,float32,1.283804,970704,2002,-52.867494,782.756120,-9999.900391,4.400000,8.530000,13.660000,23.93000,NaN


,split,is_trainable_target_3h,is_trainable_physical_3h,is_trainable_multitarget_3h,is_trainable_risk_3h,is_trainable_physical_6h,is_trainable_multitarget_6h,is_trainable_risk_6h,is_trainable_physical_12h,is_trainable_multitarget_12h,is_trainable_risk_12h,is_trainable_physical_24h,is_trainable_multitarget_24h,is_trainable_risk_24h,is_trainable_physical_48h,is_trainable_multitarget_48h,is_trainable_risk_48h
0,test,99.892021,99.892021,88.164612,99.892021,99.784035,88.070328,99.784035,99.568077,87.881737,99.568077,99.136147,87.504562,99.136147,98.272293,86.750214,98.272293
1,train,99.924004,99.924004,89.931046,99.924004,99.848007,89.862778,99.848007,99.696007,89.726242,99.696007,99.392029,89.453163,99.392029,98.818398,88.937935,98.818398
2,val,99.931320,99.931320,89.938187,99.931320,99.862633,89.876373,99.862633,99.725273,89.752747,99.725273,99.450546,89.505493,99.450546,98.901100,89.010986,98.901100


## Celda 11 — Validaciones físicas básicas

In [12]:
validation_rows = []

def add_validation(name, condition, severity="FAIL", details=""):
    validation_rows.append({"validation": name, "passed": bool(condition), "severity": severity, "details": details})

add_validation("has_rows", len(gold) > 0, details=f"rows={len(gold)}")
add_validation("has_zona_id", "zona_id" in gold.columns)
add_validation("has_timestamp", "timestamp" in gold.columns)
add_validation("has_split", "split" in gold.columns)

for horizon in HORIZONS_HOURS:
    hs_col = f"target_hs_{horizon}h"
    if hs_col in gold.columns:
        hs = pd.to_numeric(gold[hs_col], errors="coerce")
        add_validation(f"{hs_col}_exists_and_non_null", hs.notna().sum() > 0, details=f"non_null={int(hs.notna().sum())}")
        add_validation(f"{hs_col}_no_negative_values", bool((hs.dropna() >= 0).all()), details=f"min={hs.min()}")
        add_validation(f"{hs_col}_reasonable_max", bool(hs.dropna().max() <= 20), severity="WARN", details=f"max={hs.max()}")

    for dname in DIRECTION_TARGETS:
        dcol = f"target_{dname}_{horizon}h"
        if dcol in gold.columns:
            d = pd.to_numeric(gold[dcol], errors="coerce").dropna()
            add_validation(
                f"{dcol}_range_0_360",
                bool(((d >= 0) & (d <= 360)).all()),
                severity="WARN",
                details=f"min={d.min() if len(d) else None}, max={d.max() if len(d) else None}",
            )

    for risk_name in ["general", "beach", "navigation"]:
        rcol = f"target_risk_{risk_name}_{horizon}h"
        if rcol in gold.columns:
            r = pd.to_numeric(gold[rcol], errors="coerce").dropna()
            add_validation(
                f"{rcol}_range_0_3",
                bool(r.isin([0, 1, 2, 3]).all()),
                details=f"classes={sorted(r.unique().tolist()) if len(r) else []}",
            )

    scol = f"target_surf_score_{horizon}h"
    if scol in gold.columns:
        s = pd.to_numeric(gold[scol], errors="coerce").dropna()
        add_validation(
            f"{scol}_range_0_10",
            bool(((s >= 0) & (s <= 10)).all()),
            details=f"min={s.min() if len(s) else None}, max={s.max() if len(s) else None}",
        )

validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(REPORT_DIR / "quality_gold_multitarget_validations.csv", index=False)
display(validation_df)

failures = validation_df[(validation_df["severity"] == "FAIL") & (~validation_df["passed"])]
if len(failures):
    raise ValueError("Hay validaciones FAIL no superadas. Revisar quality_gold_multitarget_validations.csv")

print("Validaciones físicas básicas superadas.")

,validation,passed,severity,details
0,has_rows,True,FAIL,rows=983328
1,has_zona_id,True,FAIL,
2,has_timestamp,True,FAIL,
3,has_split,True,FAIL,
4,target_hs_3h_exists_and_non_null,True,FAIL,non_null=982524
5,target_hs_3h_no_negative_values,True,FAIL,min=0.029999999329447746
6,target_hs_3h_reasonable_max,True,WARN,max=7.829999923706055
7,target_wave_direction_3h_range_0_360,True,WARN,"min=0.0, max=359.0"
8,target_swell_direction_3h_range_0_360,True,WARN,"min=0.0, max=359.0"
9,target_wind_direction_3h_range_0_360,True,WARN,"min=0.0, max=359.0"


Validaciones físicas básicas superadas.


## Celda 12 — Crear diccionario de variables del Gold multitarget

In [13]:
data_dictionary_rows = []

for c in gold.columns:
    role = "feature"
    if c in ["zona_id", "timestamp", "split", "year", "month", "dayofweek", "hour"]:
        role = "key_or_partition"
    elif c.startswith("target_"):
        role = "target"
    elif c.startswith("is_trainable_"):
        role = "trainability_flag"
    elif c.endswith("_flag"):
        role = "quality_flag"
    elif "_lag_" in c or "_roll_" in c or "_diff_" in c:
        role = "temporal_feature"
    elif c.endswith("_sin") or c.endswith("_cos"):
        role = "circular_feature_or_target"
    elif "risk" in c:
        role = "risk_related"

    data_dictionary_rows.append({
        "column": c,
        "role": role,
        "dtype": str(gold[c].dtype),
        "missing_pct": float(gold[c].isna().mean() * 100),
        "nunique": int(gold[c].nunique(dropna=True)),
        "description": "",
    })

data_dictionary = pd.DataFrame(data_dictionary_rows).sort_values(["role", "column"])
data_dictionary.to_csv(META_DIR / "gold_multitarget_data_dictionary.csv", index=False)
display(data_dictionary.head(100))

metadata = {
    "dataset_name": "gold/multitarget_training_dataset",
    "source_dataset": str(INPUT_GOLD_DIR),
    "output_dataset": str(OUTPUT_GOLD_DIR),
    "horizons_hours": HORIZONS_HOURS,
    "target_source_map": target_source_map,
    "direction_targets": DIRECTION_TARGETS,
    "risk_thresholds": RISK_THRESHOLDS,
    "surf_score_config": SURF_SCORE_CONFIG,
    "rows": int(len(gold)),
    "columns": int(len(gold.columns)),
    "zona_count": int(gold["zona_id"].nunique()),
    "timestamp_min": str(gold["timestamp"].min()),
    "timestamp_max": str(gold["timestamp"].max()),
}

save_json(metadata, META_DIR / "gold_multitarget_metadata.json")
print("Metadata guardada:", META_DIR / "gold_multitarget_metadata.json")

,column,role,dtype,missing_pct,nunique,description
83,dayofyear_cos,circular_feature_or_target,float64,0.0,336,
82,dayofyear_sin,circular_feature_or_target,float64,0.0,325,
79,hour_cos,circular_feature_or_target,float64,0.0,22,
78,hour_sin,circular_feature_or_target,float64,0.0,22,
81,month_cos,circular_feature_or_target,float64,0.0,11,
...,...,...,...,...,...,...
67,slope_0_500m,feature,float64,0.0,10,
68,slope_500m_2km,feature,float64,0.0,12,
20,tipo_zona,feature,string,0.0,1,
27,vulnerabilidad_costera,feature,str,0.0,1,


Metadata guardada: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_metadata/gold_multitarget_metadata.json


## Celda 13 — Optimizar tipos y guardar Parquet particionado

In [14]:
# Optimización de tipos.
gold = optimize_numeric_dtypes(gold)

# Convertir strings/categorías problemáticas a string.
for c in gold.columns:
    if pd.api.types.is_object_dtype(gold[c]):
        gold[c] = gold[c].astype("string")

# Asegurar particiones.
gold["year"] = gold["timestamp"].dt.year.astype("int16")
gold["split"] = gold["split"].astype("string")
gold["isla"] = gold["isla"].astype("string").fillna("UNKNOWN")

expected_rows = len(gold)
expected_columns = len(gold.columns)

if OVERWRITE_OUTPUT and OUTPUT_GOLD_DIR.exists():
    shutil.rmtree(OUTPUT_GOLD_DIR)

OUTPUT_GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Guardando en:", OUTPUT_GOLD_DIR)
memory_report("gold final antes de guardar", gold)

# Guardado particionado.
table = pa.Table.from_pandas(gold, preserve_index=False)

pq.write_to_dataset(
    table,
    root_path=str(OUTPUT_GOLD_DIR),
    partition_cols=["split", "year"],
    compression="snappy",
    use_dictionary=True,
)

print("Guardado completado.")

# Liberar memoria antes de leer de vuelta.
del table
del gold
gc.collect()

print("Memoria liberada tras guardado.")
print("expected_rows:", expected_rows)
print("expected_columns:", expected_columns)

Guardando en: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/multitarget_training_dataset
gold final antes de guardar: shape=(983328, 669), memoria≈2484.8 MB
Guardado completado.
Memoria liberada tras guardado.
expected_rows: 983328
expected_columns: 669


## Celda 14 — Leer de vuelta y validar output

In [15]:
dataset_out = ds.dataset(str(OUTPUT_GOLD_DIR), format="parquet", partitioning="hive")
table_out = dataset_out.to_table()
test = table_out.to_pandas()

del table_out
gc.collect()

test["timestamp"] = ensure_utc(test["timestamp"])

print("Dataset leído de vuelta:")
memory_report("test", test)

print("Rango temporal:")
print(test["timestamp"].min(), "→", test["timestamp"].max())

print("Zonas:", test["zona_id"].nunique())
print("Columnas:", len(test.columns))

# Validaciones de lectura.
if len(test) != expected_rows:
    raise ValueError(f"Row count no coincide: guardado={len(test)}, memoria={expected_rows}")

required_core_cols = [
    "zona_id",
    "timestamp",
    "split",
    "target_hs_3h",
    "target_hs_6h",
    "target_hs_12h",
    "target_hs_24h",
    "target_risk_beach_24h",
    "target_risk_navigation_24h",
    "target_surf_score_24h",
]

missing_required = [c for c in required_core_cols if c not in test.columns]

if missing_required:
    raise ValueError("Faltan columnas core: " + json.dumps(missing_required, indent=2))

print("Validación de lectura OK.")
display(test.head())

Dataset leído de vuelta:
test: shape=(983328, 669), memoria≈2485.4 MB
Rango temporal:
2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
Zonas: 14
Columnas: 669
Validación de lectura OK.


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,is_trainable_multitarget_12h,is_trainable_risk_12h,is_trainable_physical_24h,is_trainable_multitarget_24h,is_trainable_risk_24h,is_trainable_physical_48h,is_trainable_multitarget_48h,is_trainable_risk_48h,split,year
0,2024-01-01 00:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166666,-14.0,0.14,NaN,14.660000,2.49,129,0.06,...,False,True,True,False,True,True,False,True,test,2024
1,2024-01-01 01:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166666,-14.0,0.14,NaN,14.660000,2.57,270,0.06,...,False,True,True,False,True,True,False,True,test,2024
2,2024-01-01 02:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166666,-14.0,0.14,NaN,14.660000,2.63,283,0.07,...,False,True,True,False,True,True,False,True,test,2024
3,2024-01-01 03:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166666,-14.0,0.15,NaN,14.660000,2.53,303,0.07,...,False,True,True,False,True,True,False,True,test,2024
4,2024-01-01 04:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166666,-14.0,0.17,NaN,16.120001,2.30,32,0.08,...,False,True,True,False,True,True,False,True,test,2024


## Celda 15 — Reporte ejecutivo final

In [16]:
executive_summary = {
    "dataset": "gold/multitarget_training_dataset",
    "rows": int(len(test)),
    "columns": int(len(test.columns)),
    "zones": int(test["zona_id"].nunique()),
    "timestamp_min": str(test["timestamp"].min()),
    "timestamp_max": str(test["timestamp"].max()),
    "horizons_hours": HORIZONS_HOURS,
    "physical_targets_available": sorted(list(target_source_map.keys())),
    "derived_targets": [
        "target_risk_general_{h}h",
        "target_risk_beach_{h}h",
        "target_risk_navigation_{h}h",
        "target_surf_score_{h}h",
        "target_surf_quality_{h}h",
    ],
    "main_output_dir": str(OUTPUT_GOLD_DIR),
    "reports": {
        "target_summary": str(REPORT_DIR / "quality_gold_multitarget_target_summary.csv"),
        "trainability_summary": str(REPORT_DIR / "quality_gold_multitarget_trainability_summary.csv"),
        "validations": str(REPORT_DIR / "quality_gold_multitarget_validations.csv"),
        "data_dictionary": str(META_DIR / "gold_multitarget_data_dictionary.csv"),
    },
}

save_json(executive_summary, META_DIR / "gold_multitarget_executive_summary.json")

summary_md = f"""
# Gold multitarget dataset — DeepWave Canarias

Se ha generado el dataset:

```text
{OUTPUT_GOLD_DIR}
```

## Contenido

- Filas: {len(test):,}
- Columnas: {len(test.columns):,}
- Zonas: {test["zona_id"].nunique():,}
- Periodo: {test["timestamp"].min()} → {test["timestamp"].max()}
- Horizontes: {HORIZONS_HOURS}

## Targets físicos disponibles

{", ".join(sorted(list(target_source_map.keys())))}

## Targets derivados

Para cada horizonte se han creado:

```text
target_risk_general
target_risk_beach
target_risk_navigation
target_surf_score
target_surf_quality
```

## Uso recomendado

Este dataset sirve para los siguientes notebooks:

```text
15_model_training_multitarget_physical.ipynb
16_model_training_risk_modules.ipynb
17_model_surf_score.ipynb
```

## Nota metodológica

Los riesgos diferenciados y el surf score son variables derivadas mediante reglas físicas interpretables.
No sustituyen observaciones oficiales de banderas, rescates o avisos marítimos.
"""

(REPORT_DIR / "gold_multitarget_executive_summary.md").write_text(summary_md, encoding="utf-8")
print(summary_md)


# Gold multitarget dataset — DeepWave Canarias

Se ha generado el dataset:

```text
/Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/multitarget_training_dataset
```

## Contenido

- Filas: 983,328
- Columnas: 669
- Zonas: 14
- Periodo: 2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
- Horizontes: [3, 6, 12, 24, 48]

## Targets físicos disponibles

current_direction, current_speed, daily_tidal_range, hs, sea_level, sea_surface_salinity, sea_surface_temperature, swell_direction, swell_height, swell_period, tm02, tp, u10, v10, wave_direction, wind_direction, wind_speed, wind_wave_height

## Targets derivados

Para cada horizonte se han creado:

```text
target_risk_general
target_risk_beach
target_risk_navigation
target_surf_score
target_surf_quality
```

## Uso recomendado

Este dataset sirve para los siguientes notebooks:

```text
15_model_training_multitarget_physical.ipynb
16_model_training_risk_modules.ipynb
17_model_surf_score.ipynb
```

## Nota met

## Celda 16 — Validación final

In [19]:
required_files = [
    META_DIR / "gold_multitarget_metadata.json",
    META_DIR / "gold_multitarget_executive_summary.json",
    META_DIR / "gold_multitarget_data_dictionary.csv",
    REPORT_DIR / "quality_gold_multitarget_target_summary.csv",
    REPORT_DIR / "quality_gold_multitarget_trainability_summary.csv",
    REPORT_DIR / "quality_gold_multitarget_validations.csv",
    REPORT_DIR / "gold_multitarget_executive_summary.md",
]

missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Faltan archivos de salida: " + json.dumps(missing_files, indent=2))

if not OUTPUT_GOLD_DIR.exists():
    raise FileNotFoundError("No existe el directorio de salida Gold multitarget.")

parquet_files = list(OUTPUT_GOLD_DIR.rglob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError("No se encontraron Parquet en gold/multitarget_training_dataset.")

print("Archivos Parquet generados:", len(parquet_files))
print("Salida:", OUTPUT_GOLD_DIR)
print("Reportes:")
for p in required_files:
    print("-", p)

print("✅ Gold multitarget dataset generado correctamente.")

Archivos Parquet generados: 11
Salida: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/multitarget_training_dataset
Reportes:
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_metadata/gold_multitarget_metadata.json
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_metadata/gold_multitarget_executive_summary.json
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_metadata/gold_multitarget_data_dictionary.csv
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_quality_reports/quality_gold_multitarget_target_summary.csv
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_quality_reports/quality_gold_multitarget_trainability_summary.csv
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/_quality_reports/quality_gold_multitarget_validations.csv
- /Users/rauljimenez/Development/Projects/AI-Projects/

## Resultado esperado

Al final debe aparecer:

```text
✅ Gold multitarget dataset generado correctamente.
```

Salida principal:

```text
gold/multitarget_training_dataset/
```

Reportes principales:

```text
gold/_quality_reports/quality_gold_multitarget_target_summary.csv
gold/_quality_reports/quality_gold_multitarget_trainability_summary.csv
gold/_quality_reports/quality_gold_multitarget_validations.csv
gold/_quality_reports/gold_multitarget_executive_summary.md
gold/_metadata/gold_multitarget_data_dictionary.csv
gold/_metadata/gold_multitarget_metadata.json
```

Siguientes notebooks recomendados:

```text
15_model_training_multitarget_physical.ipynb
16_model_training_risk_modules.ipynb
17_model_surf_score.ipynb
```

## Ejecución local en Mac

Antes de ejecutar, revisa la celda 2:

```python
LOCAL_BASE_DIR = None
```

Si el proyecto no se detecta automáticamente, pon la ruta exacta:

```python
LOCAL_BASE_DIR = Path("/Users/tu_usuario/Documents/AI Projects/DeepWave Canarias")
```

Si tienes 16 GB de RAM, deja:

```python
MAC_SAFE_MODE = True
```

Si tienes 32 GB o más y quieres generar más features, puedes probar:

```python
MAC_SAFE_MODE = False
```

pero es recomendable ejecutar primero en modo seguro.
